# 2036 프리모템: AI 신약설계의 화학공간 붕괴 — 재현 노트북

**2026 KAIST AI x 실패 아이디어 공모전** 제출작 보완자료

이 노트북은 제안서와 보완 슬라이드에 인용된 모든 수치를 원자료로부터 직접 재계산합니다.
별도 설치나 클론 없이 이 셀들을 순서대로 실행하면 됩니다 (Runtime → Run all).

- GitHub 저장소: https://github.com/yenakim2004/amr-2036-premortem
- 데이터 출처: WHO Antibacterial pipeline analysis (2023, 2025); Naghavi et al., *The Lancet* 2024 (GRAM Project)
- 이 노트북이 계산하는 것: (1) WHO 실측 파이프라인 축소율 → 2036년 외삽, (2) 외삽 민감도 분석, (3) 계열(class) 편중 확인


In [ ]:
# 1. 원자료 CSV 불러오기 (공개 GitHub 저장소에서 직접 fetch)
import pandas as pd
import numpy as np

RAW_BASE = "https://raw.githubusercontent.com/yenakim2004/amr-2036-premortem/main/data"

who = pd.read_csv(f"{RAW_BASE}/who_pipeline.csv")
entrants = pd.read_csv(f"{RAW_BASE}/new_entrants_class_composition.csv")
mortality = pd.read_csv(f"{RAW_BASE}/gram_amr_mortality.csv")
cum_forecast = pd.read_csv(f"{RAW_BASE}/gram_amr_cumulative_forecast.csv")
scaffold = pd.read_csv(f"{RAW_BASE}/scaffold_coverage_benchmark.csv")

print("WHO 파이프라인 실측치:")
who


## 2. 2023 → 2025 실측치 기반 연간 감소율

WHO 임상 항생제 후보 파이프라인은 2023년 97개 → 2025년 90개로 감소했고,
그중 "혁신적(innovative)" 후보는 2025년 기준 15개(16.7%)였습니다.


In [ ]:
def project_pipeline_to_2036(who_df: pd.DataFrame, target_year: int = 2036) -> dict:
    """WHO 2023->2025 실측 파이프라인 규모로부터 연간 감소율을 구하고,
    2025년 혁신 후보 비율(16.7%)을 고정한 채 target_year까지 외삽한다.
    이 외삽은 model/mode collapse로 인한 추가적인 다양성 손실을 반영하지
    않은 보수적 추정치임을 명시한다.
    """
    n_2023 = who_df.loc[who_df.year == 2023, "total_clinical_candidates"].iloc[0]
    n_2025 = who_df.loc[who_df.year == 2025, "total_clinical_candidates"].iloc[0]
    innov_frac_2025 = who_df.loc[who_df.year == 2025, "innovative_fraction"].iloc[0]

    annual_rate = (n_2025 / n_2023) ** (1 / 2) - 1  # 2-year basis clean CAGR
    total_proj = n_2025 * (1 + annual_rate) ** (target_year - 2025)
    innov_proj = total_proj * innov_frac_2025

    return {
        "annual_rate": annual_rate,
        "total_pipeline_proj": total_proj,
        "innovative_candidates_proj": innov_proj,
        "innovative_fraction_held": innov_frac_2025,
    }

proj = project_pipeline_to_2036(who)
print(f"연간 감소율 (2023->2025 실측 기준): {proj['annual_rate']:.4%}")
print(f"2036년 전체 파이프라인 추정: {proj['total_pipeline_proj']:.1f}개")
print(f"2036년 혁신 후보 추정: {proj['innovative_candidates_proj']:.1f}개 "
      f"(2025년 혁신 비율 {proj['innovative_fraction_held']:.1%} 고정)")


## 3. 민감도 분석 — 연간 감소율 ±2%p

외삽이 지나치게 단순하다는 반론에 대응하기 위해, 연간 감소율 가정을
실측 기준값 주변 ±2 퍼센트포인트 범위에서 0.5%p 간격으로 흔들어
2036년 결과가 어떻게 달라지는지 확인합니다.


In [ ]:
def sensitivity_analysis(who_df: pd.DataFrame, target_year: int = 2036,
                          delta_pp_range=(-2, -1.5, -1, -0.5, 0, 0.5, 1, 1.5, 2)) -> pd.DataFrame:
    n_2025 = who_df.loc[who_df.year == 2025, "total_clinical_candidates"].iloc[0]
    innov_frac_2025 = who_df.loc[who_df.year == 2025, "innovative_fraction"].iloc[0]
    base = project_pipeline_to_2036(who_df, target_year)
    rate_center = base["annual_rate"]

    rows = []
    for d_pp in delta_pp_range:
        r = rate_center + d_pp / 100
        tp = n_2025 * (1 + r) ** (target_year - 2025)
        ip = tp * innov_frac_2025
        rows.append({
            "delta_pp": d_pp,
            "annual_rate_pct": r * 100,
            "total_pipeline_2036": tp,
            "innovative_candidates_2036": ip,
        })
    return pd.DataFrame(rows)

sens = sensitivity_analysis(who)
sens.round(2)


In [ ]:
# 4. 민감도 분석 시각화
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].plot(sens["delta_pp"], sens["total_pipeline_2036"], marker="o", color="#3b6fa0")
axes[0].axhline(90, ls="--", color="#888888", lw=1, label="2025년 실측치 (90개)")
axes[0].set_xlabel("연간 감소율 조정 (%p)")
axes[0].set_ylabel("2036년 전체 파이프라인 (개)")
axes[0].set_title("전체 파이프라인 민감도")
axes[0].legend(fontsize=8)

axes[1].plot(sens["delta_pp"], sens["innovative_candidates_2036"], marker="o", color="#c1633a")
axes[1].axhline(15, ls="--", color="#888888", lw=1, label="2025년 실측치 (15개)")
axes[1].set_xlabel("연간 감소율 조정 (%p)")
axes[1].set_ylabel("2036년 혁신 후보 (개)")
axes[1].set_title("혁신 후보 민감도")
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()

print("가장 낙관적 시나리오(+2%p)에서도 혁신 후보 추정:",
      round(sens.loc[sens.delta_pp == 2.0, 'innovative_candidates_2036'].iloc[0], 1),
      "개 — 2025년 실측치(15개)에 못 미침")


## 5. 신규 진입 후보의 계열(class) 편중

WHO/*Lancet Microbe* (2024) 분석에 따르면 2024년 신규 진입 후보 13개 중
10개는 기존 계열, 3개만 신규 계열이었습니다 — 재귀적 재학습이 화학공간을
좁힐 때 예상되는 패턴과 일치합니다.


In [ ]:
entrants


## 6. AMR 사망 부담 전망 (Naghavi et al., *The Lancet* 2024, GRAM Project)

AMR 직접 기인 사망은 2021년 114만 명(실측)에서 2050년 191만 명(전망)으로 증가할 것으로
추정되며, 2025–2050년 누적 사망은 3,900만 명으로 전망됩니다.


In [ ]:
print(mortality)
print()
print(cum_forecast)


## 출처 및 한계

- WHO, *Antibacterial agents in clinical and preclinical development* (2023, 2025 editions)
- WHO / *Lancet Microbe*, 신규 진입 항생제 계열 구성 분석 (2024)
- Naghavi, M. et al. (2024). Global burden of bacterial antimicrobial resistance 1990–2021. **The Lancet**, GRAM Project.
- Nature (2025). "Four ways to power-up AI for drug discovery." DOI: 10.1038/d41586-025-00602-5
- Drug Discovery World (2025). Sheffield 대학 의약화학자 인터뷰 연구 소개.
- Shumailov, I. et al. Model collapse 관련 연구 (recursive self-training diversity loss)

**한계**: 위 외삽은 선형(고정 CAGR) 모델이며 model/mode collapse로 인한 추가 다양성
손실을 반영하지 않은 보수적 추정치입니다. 상세 방법론과 원자료는
[GitHub 저장소](https://github.com/yenakim2004/amr-2036-premortem)를 참고하십시오.
